In [4]:
# ============================================
# DATA EXPLORATION - Resume Dataset
# Run this FIRST to understand the data structure
# ============================================

import pandas as pd
import numpy as np
import os

print("="*60)
print("📂 CHECKING DATASET LOCATION")
print("="*60)

# Check what files exist in the input directory
input_path = '/kaggle/input/'
print(f"\nContents of {input_path}:")
for root, dirs, files in os.walk(input_path):
    level = root.replace(input_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit to first 10 files per folder
        print(f"{subindent}{file}")
    if len(files) > 10:
        print(f"{subindent}... and {len(files) - 10} more files")

print("\n" + "="*60)
print("📊 LOADING AND EXPLORING RESUME DATA")
print("="*60)

# Try different possible paths
possible_paths = [
    '/kaggle/input/resume-dataset/Resume.csv',
    '/kaggle/input/resume-dataset/Resume/Resume.csv',
    '/kaggle/input/resume-dataset/resume.csv',
    '/kaggle/input/resume-dataset/UpdatedResumeDataSet.csv'
]

df = None
used_path = None

for path in possible_paths:
    if os.path.exists(path):
        print(f"\n✅ Found file at: {path}")
        df = pd.read_csv(path)
        used_path = path
        break
    else:
        print(f"❌ Not found: {path}")

if df is None:
    # List all CSV files in the input directory
    print("\n🔍 Searching for all CSV files...")
    for root, dirs, files in os.walk(input_path):
        for file in files:
            if file.endswith('.csv'):
                full_path = os.path.join(root, file)
                print(f"Found CSV: {full_path}")
                # Try to load the first CSV found
                if df is None:
                    df = pd.read_csv(full_path)
                    used_path = full_path
                    print(f"✅ Loaded: {full_path}")

if df is None:
    print("\n❌ ERROR: Could not find any CSV file!")
    print("Please check that you added the 'Resume Dataset' to this notebook.")
else:
    print(f"\n📁 Using file: {used_path}")
    
    print("\n" + "-"*60)
    print("BASIC INFO")
    print("-"*60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn names: {df.columns.tolist()}")
    print(f"\nColumn data types:")
    print(df.dtypes)
    
    print("\n" + "-"*60)
    print("MISSING VALUES")
    print("-"*60)
    print(df.isnull().sum())
    
    print("\n" + "-"*60)
    print("FIRST 3 ROWS (truncated)")
    print("-"*60)
    pd.set_option('display.max_colwidth', 100)
    print(df.head(3))
    
    print("\n" + "-"*60)
    print("COLUMN DETAILS")
    print("-"*60)
    
    for col in df.columns:
        print(f"\n📌 Column: '{col}'")
        print(f"   Type: {df[col].dtype}")
        print(f"   Non-null: {df[col].notna().sum()}/{len(df)}")
        print(f"   Unique values: {df[col].nunique()}")
        
        if df[col].dtype == 'object':
            # Show sample values
            sample_vals = df[col].dropna().head(2).tolist()
            for i, val in enumerate(sample_vals):
                val_str = str(val)[:200] + "..." if len(str(val)) > 200 else str(val)
                print(f"   Sample {i+1}: {val_str}")
        else:
            print(f"   Min: {df[col].min()}, Max: {df[col].max()}")
    
    # Check for Category column specifically
    print("\n" + "-"*60)
    print("CATEGORY DISTRIBUTION (if exists)")
    print("-"*60)
    
    category_col = None
    for col in df.columns:
        if 'category' in col.lower() or 'label' in col.lower() or 'class' in col.lower():
            category_col = col
            break
    
    if category_col:
        print(f"\nUsing column: '{category_col}'")
        print(f"\nValue counts:")
        print(df[category_col].value_counts())
    else:
        print("No category/label column found")
    
    # Check for resume text column
    print("\n" + "-"*60)
    print("RESUME TEXT COLUMN (if exists)")
    print("-"*60)
    
    text_col = None
    for col in df.columns:
        if 'resume' in col.lower() or 'text' in col.lower() or 'content' in col.lower():
            if df[col].dtype == 'object':
                text_col = col
                break
    
    if text_col:
        print(f"\nUsing column: '{text_col}'")
        print(f"Average length: {df[text_col].str.len().mean():.0f} characters")
        print(f"Min length: {df[text_col].str.len().min()}")
        print(f"Max length: {df[text_col].str.len().max()}")
        print(f"\nSample resume (first 500 chars):")
        print(df[text_col].iloc[0][:500])
    else:
        print("No resume text column found")

print("\n" + "="*60)
print("✅ EXPLORATION COMPLETE")
print("="*60)
print("\n👆 Copy ALL the output above and share it with me!")

📂 CHECKING DATASET LOCATION

Contents of /kaggle/input/:
/
resume-dataset/
  Resume/
    Resume.csv
  data/
    data/
      DESIGNER/
        22506245.pdf
        13998435.pdf
        67582956.pdf
        34349255.pdf
        26790545.pdf
        12674307.pdf
        11807040.pdf
        44145704.pdf
        27497542.pdf
        11155153.pdf
        ... and 97 more files
      BPO/
        95625660.pdf
        15573418.pdf
        57706851.pdf
        16492045.pdf
        19362586.pdf
        63158213.pdf
        38707449.pdf
        13964744.pdf
        27710853.pdf
        23933031.pdf
        ... and 12 more files
      FINANCE/
        31948488.pdf
        86595128.pdf
        11877150.pdf
        19243556.pdf
        24530382.pdf
        26767199.pdf
        15224503.pdf
        25678238.pdf
        34250007.pdf
        19540089.pdf
        ... and 108 more files
      CONSTRUCTION/
        12890045.pdf
        45475027.pdf
        16353584.pdf
        24953126.pdf
        5652573

In [6]:
# ============================================
# NOTEBOOK 1: Resume & ATS Models
# ============================================
# Dataset: Resume Dataset (2,484 resumes, 24 categories)
# File: /kaggle/input/resume-dataset/Resume/Resume.csv
# Columns: ID, Resume_str, Resume_html, Category
# ============================================

import pandas as pd
import numpy as np
import json
import re
import pickle
import subprocess
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ============================================
# SETUP: Install and load spaCy
# ============================================

print("📦 Installing spaCy model...")
subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], 
               check=True, capture_output=True)

import spacy
nlp = spacy.load('en_core_web_sm')
print("✅ spaCy loaded")

# ============================================
# PART 1: Load Resume Dataset
# ============================================

print("\n" + "="*60)
print("PART 1: Loading Resume Dataset")
print("="*60)

# Load data from correct path
df = pd.read_csv('/kaggle/input/resume-dataset/Resume/Resume.csv')

print(f"✅ Loaded {len(df)} resumes")
print(f"✅ Categories: {df['Category'].nunique()}")
print(f"\nCategory distribution:")
print(df['Category'].value_counts())

# ============================================
# PART 2: Clean Resume Text
# ============================================

print("\n" + "="*60)
print("PART 2: Cleaning Resume Text")
print("="*60)

def clean_text(text):
    """Clean resume text for processing"""
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove extra whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s\.\,\-\(\)\@\+]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r' +', ' ', text)
    
    return text.strip()

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

# Show sample
print(f"✅ Cleaned {len(df)} resumes")
print(f"\nSample cleaned resume (first 300 chars):")
print(df['cleaned_resume'].iloc[0][:300])

# Text statistics
df['word_count'] = df['cleaned_resume'].apply(lambda x: len(x.split()))
print(f"\nWord count statistics:")
print(df['word_count'].describe())

# ============================================
# PART 3: Train Resume Category Classifier
# ============================================

print("\n" + "="*60)
print("PART 3: Training Resume Category Classifier")
print("="*60)

X = df['cleaned_resume']
y = df['Category']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# TF-IDF Vectorization
print("\n🔄 Building TF-IDF features...")
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    stop_words='english',
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"✅ TF-IDF shape: {X_train_tfidf.shape}")

# Train Logistic Regression classifier
print("\n🔄 Training classifier...")
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
    C=1.0,
    class_weight='balanced'  # Handle class imbalance
)
classifier.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = classifier.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Category Classifier Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# Save models
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('category_classifier.pkl', 'wb') as f:
    pickle.dump(classifier, f)

print("✅ Saved: tfidf_vectorizer.pkl")
print("✅ Saved: category_classifier.pkl")

# ============================================
# PART 4: Extract Skills from Resumes
# ============================================

print("\n" + "="*60)
print("PART 4: Extracting Skills Taxonomy")
print("="*60)

# Comprehensive skill list
SKILL_PATTERNS = [
    # Programming Languages
    'python', 'java', 'javascript', 'c\\+\\+', 'c#', 'ruby', 'php', 'swift', 
    'kotlin', 'r', 'scala', 'go', 'rust', 'typescript', 'sql', 'html', 'css',
    'perl', 'matlab', 'bash', 'shell', 'powershell', 'vba', 'objective-c',
    
    # Web Frameworks & Libraries
    'react', 'angular', 'vue', 'django', 'flask', 'fastapi', 'spring', 
    'node\\.js', 'express', 'jquery', 'bootstrap', 'asp\\.net', 'laravel',
    'ruby on rails', 'next\\.js', 'gatsby',
    
    # Data Science & ML
    'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'sklearn', 'pandas', 
    'numpy', 'scipy', 'matplotlib', 'seaborn', 'plotly', 'opencv',
    'machine learning', 'deep learning', 'neural network', 'nlp', 
    'natural language processing', 'computer vision', 'data mining',
    'data analysis', 'data visualization', 'statistics', 'predictive modeling',
    'regression', 'classification', 'clustering', 'time series',
    
    # Big Data & Databases
    'mysql', 'postgresql', 'mongodb', 'redis', 'cassandra', 'oracle', 
    'sqlite', 'sql server', 'dynamodb', 'elasticsearch', 'neo4j',
    'hadoop', 'spark', 'hive', 'kafka', 'airflow', 'etl', 'data warehouse',
    'snowflake', 'redshift', 'bigquery',
    
    # Cloud & DevOps
    'aws', 'amazon web services', 'azure', 'gcp', 'google cloud', 
    'docker', 'kubernetes', 'jenkins', 'terraform', 'ansible',
    'ci/cd', 'git', 'github', 'gitlab', 'bitbucket', 'linux', 'unix',
    'nginx', 'apache', 'serverless', 'microservices', 'rest api', 'graphql',
    
    # Business & Analytics Tools
    'excel', 'power bi', 'tableau', 'looker', 'qlik', 'sap', 'salesforce',
    'google analytics', 'adobe analytics', 'spss', 'sas', 'stata',
    
    # Project Management & Methodologies
    'agile', 'scrum', 'kanban', 'jira', 'trello', 'asana', 'confluence',
    'project management', 'product management', 'waterfall', 'lean',
    
    # Soft Skills
    'communication', 'leadership', 'teamwork', 'problem solving', 
    'critical thinking', 'time management', 'presentation',
    
    # Domain Specific
    'financial analysis', 'accounting', 'budgeting', 'forecasting',
    'marketing', 'seo', 'social media', 'content management',
    'customer service', 'sales', 'negotiation', 'crm',
    'healthcare', 'hipaa', 'clinical', 'patient care',
    'legal', 'compliance', 'regulatory', 'contract',
    'engineering', 'cad', 'autocad', 'solidworks', 'mechanical',
    'electrical', 'civil', 'structural',
]

def extract_skills(text):
    """Extract skills from resume text using pattern matching"""
    text_lower = text.lower()
    found_skills = []
    
    for skill in SKILL_PATTERNS:
        # Use word boundaries for accurate matching
        pattern = r'\b' + skill + r'\b'
        if re.search(pattern, text_lower):
            # Normalize skill name (remove regex escapes)
            clean_skill = skill.replace('\\', '')
            found_skills.append(clean_skill)
    
    return list(set(found_skills))  # Remove duplicates

# Extract skills from all resumes
print("🔄 Extracting skills from resumes...")
df['extracted_skills'] = df['cleaned_resume'].apply(extract_skills)
df['skill_count'] = df['extracted_skills'].apply(len)

# Get all unique skills
all_skills = set()
for skills in df['extracted_skills']:
    all_skills.update(skills)

print(f"✅ Found {len(all_skills)} unique skills across all resumes")
print(f"Average skills per resume: {df['skill_count'].mean():.1f}")

# Build skill taxonomy (skill -> categories and frequency)
skill_taxonomy = {}
for skill in all_skills:
    # Find which resumes have this skill
    mask = df['extracted_skills'].apply(lambda x: skill in x)
    matching_df = df[mask]
    
    # Get category distribution
    category_counts = matching_df['Category'].value_counts().to_dict()
    
    skill_taxonomy[skill] = {
        'frequency': int(mask.sum()),
        'categories': list(category_counts.keys()),
        'category_counts': category_counts
    }

# Sort by frequency
skill_taxonomy = dict(sorted(
    skill_taxonomy.items(), 
    key=lambda x: x[1]['frequency'], 
    reverse=True
))

# Show top skills
print("\nTop 15 skills by frequency:")
for i, (skill, data) in enumerate(list(skill_taxonomy.items())[:15]):
    print(f"  {i+1}. {skill}: {data['frequency']} resumes")

# Save taxonomy
with open('skill_taxonomy.json', 'w') as f:
    json.dump(skill_taxonomy, f, indent=2)

print("\n✅ Saved: skill_taxonomy.json")

# ============================================
# PART 5: Build ATS Scoring System
# ============================================

print("\n" + "="*60)
print("PART 5: Building ATS Scoring System")
print("="*60)

def calculate_ats_features(row):
    """Calculate ATS-relevant features from a resume"""
    
    text = row['cleaned_resume']
    skills = row['extracted_skills']
    word_count = row['word_count']
    
    # Feature 1: Resume length score (optimal: 400-800 words)
    if 400 <= word_count <= 800:
        length_score = 1.0
    elif 300 <= word_count < 400 or 800 < word_count <= 1000:
        length_score = 0.75
    elif 200 <= word_count < 300 or 1000 < word_count <= 1200:
        length_score = 0.5
    else:
        length_score = 0.3
    
    # Feature 2: Skills density score
    skill_count = len(skills)
    # Optimal: 10-20 skills
    if 10 <= skill_count <= 20:
        skill_score = 1.0
    elif 5 <= skill_count < 10 or 20 < skill_count <= 25:
        skill_score = 0.75
    elif skill_count < 5:
        skill_score = skill_count / 5  # 0 to 1
    else:
        skill_score = 0.6  # Too many skills might indicate keyword stuffing
    
    # Feature 3: Section completeness
    text_lower = text.lower()
    sections = {
        'experience': r'\b(experience|employment|work history|professional background)\b',
        'education': r'\b(education|degree|university|college|bachelor|master|phd)\b',
        'skills': r'\b(skills|technical skills|competencies|proficiencies)\b',
        'summary': r'\b(summary|objective|profile|about)\b',
        'contact': r'\b(email|phone|address|linkedin|contact)\b'
    }
    
    section_matches = sum(1 for pattern in sections.values() 
                         if re.search(pattern, text_lower))
    section_score = section_matches / len(sections)
    
    # Feature 4: Contact information
    has_email = bool(re.search(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', text))
    has_phone = bool(re.search(
        r'\b(\+\d{1,3}[-.]?)?\(?\d{3}\)?[-.]?\d{3}[-.]?\d{4}\b', text))
    has_linkedin = bool(re.search(r'linkedin', text_lower))
    contact_score = (has_email + has_phone + has_linkedin) / 3
    
    # Feature 5: Keyword density (skills per 100 words)
    if word_count > 0:
        density = (skill_count / word_count) * 100
        # Optimal: 2-5 skills per 100 words
        if 2 <= density <= 5:
            density_score = 1.0
        elif 1 <= density < 2 or 5 < density <= 7:
            density_score = 0.75
        else:
            density_score = 0.5
    else:
        density_score = 0
    
    # Feature 6: Action verbs (indicates achievement-oriented writing)
    action_verbs = [
        'achieved', 'managed', 'developed', 'created', 'implemented', 
        'increased', 'reduced', 'improved', 'led', 'designed', 
        'built', 'launched', 'delivered', 'generated', 'established'
    ]
    action_count = sum(1 for verb in action_verbs if verb in text_lower)
    action_score = min(action_count / 10, 1.0)
    
    return {
        'length_score': length_score,
        'skill_score': skill_score,
        'section_score': section_score,
        'contact_score': contact_score,
        'density_score': density_score,
        'action_score': action_score,
        'word_count': word_count,
        'skill_count': skill_count
    }

# Calculate features for all resumes
print("🔄 Calculating ATS features for all resumes...")
ats_features_list = []
for idx, row in df.iterrows():
    features = calculate_ats_features(row)
    ats_features_list.append(features)

ats_df = pd.DataFrame(ats_features_list)

# Calculate weighted ATS score
weights = {
    'length_score': 0.10,
    'skill_score': 0.30,
    'section_score': 0.20,
    'contact_score': 0.10,
    'density_score': 0.15,
    'action_score': 0.15
}

ats_df['ats_score'] = (
    ats_df['length_score'] * weights['length_score'] +
    ats_df['skill_score'] * weights['skill_score'] +
    ats_df['section_score'] * weights['section_score'] +
    ats_df['contact_score'] * weights['contact_score'] +
    ats_df['density_score'] * weights['density_score'] +
    ats_df['action_score'] * weights['action_score']
) * 100

# Create pass/fail label (threshold: 60)
ats_df['ats_pass'] = (ats_df['ats_score'] >= 60).astype(int)

print(f"\n📊 ATS Score Statistics:")
print(f"Mean Score: {ats_df['ats_score'].mean():.1f}")
print(f"Std Dev: {ats_df['ats_score'].std():.1f}")
print(f"Min: {ats_df['ats_score'].min():.1f}")
print(f"Max: {ats_df['ats_score'].max():.1f}")
print(f"\nPass Rate (score >= 60): {ats_df['ats_pass'].mean()*100:.1f}%")

print(f"\nScore Distribution:")
print(ats_df['ats_score'].describe())

# Train ATS classifier
print("\n🔄 Training ATS classifier...")
feature_cols = ['length_score', 'skill_score', 'section_score', 
                'contact_score', 'density_score', 'action_score']

X_ats = ats_df[feature_cols]
y_ats = ats_df['ats_pass']

X_ats_train, X_ats_test, y_ats_train, y_ats_test = train_test_split(
    X_ats, y_ats, test_size=0.2, random_state=42, stratify=y_ats
)

ats_classifier = LogisticRegression(random_state=42, max_iter=500)
ats_classifier.fit(X_ats_train, y_ats_train)

# Evaluate
y_ats_pred = ats_classifier.predict(X_ats_test)
ats_accuracy = accuracy_score(y_ats_test, y_ats_pred)

print(f"✅ ATS Classifier Accuracy: {ats_accuracy:.3f} ({ats_accuracy*100:.1f}%)")

# Save ATS model
with open('ats_scorer.pkl', 'wb') as f:
    pickle.dump(ats_classifier, f)

# Save ATS configuration
ats_config = {
    'feature_columns': feature_cols,
    'weights': weights,
    'pass_threshold': 60,
    'score_range': {'min': 0, 'max': 100}
}

with open('ats_config.json', 'w') as f:
    json.dump(ats_config, f, indent=2)

print("✅ Saved: ats_scorer.pkl")
print("✅ Saved: ats_config.json")

# ============================================
# PART 6: Save Combined Data for Reference
# ============================================

print("\n" + "="*60)
print("PART 6: Saving Reference Data")
print("="*60)

# Save category list
categories = df['Category'].unique().tolist()
with open('categories.json', 'w') as f:
    json.dump(categories, f, indent=2)

print(f"✅ Saved: categories.json ({len(categories)} categories)")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*60)
print("✅ NOTEBOOK 1 COMPLETE")
print("="*60)

print("\n📁 Generated Files:")
print("   1. tfidf_vectorizer.pkl - TF-IDF vectorizer for resume text")
print("   2. category_classifier.pkl - Resume category classifier (24 classes)")
print("   3. skill_taxonomy.json - Skill → category mapping with frequencies")
print("   4. ats_scorer.pkl - ATS pass/fail classifier")
print("   5. ats_config.json - ATS scoring configuration & weights")
print("   6. categories.json - List of all job categories")

print("\n📊 Model Performance:")
print(f"   Category Classifier: {accuracy*100:.1f}% accuracy")
print(f"   ATS Classifier: {ats_accuracy*100:.1f}% accuracy")

print("\n📈 Dataset Statistics:")
print(f"   Total resumes: {len(df)}")
print(f"   Categories: {df['Category'].nunique()}")
print(f"   Unique skills found: {len(all_skills)}")
print(f"   Avg skills per resume: {df['skill_count'].mean():.1f}")
print(f"   Avg ATS score: {ats_df['ats_score'].mean():.1f}/100")

print("\n📥 Download all 6 files and place in 'trained_models/' folder")
print("="*60)

📦 Installing spaCy model...
✅ spaCy loaded

PART 1: Loading Resume Dataset
✅ Loaded 2484 resumes
✅ Categories: 24

Category distribution:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

PART 2: Cleaning Resume Text
✅ Cleaned 2484 resumes

Sample cleaned resume (first 300 chars):
HR ADMINISTR

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Found 143 unique skills across all resumes
Average skills per resume: 6.8

Top 15 skills by frequency:
  1. sales: 1311 resumes
  2. communication: 1297 resumes
  3. leadership: 1026 resumes
  4. customer service: 974 resumes
  5. excel: 960 resumes
  6. marketing: 870 resumes
  7. compliance: 772 resumes
  8. accounting: 560 resumes
  9. project management: 556 resumes
  10. engineering: 437 resumes
  11. problem solving: 426 resumes
  12. contract: 395 resumes
  13. time management: 377 resumes
  14. presentation: 353 resumes
  15. budgeting: 305 resumes

✅ Saved: skill_taxonomy.json

PART 5: Building ATS Scoring System
🔄 Calculating ATS features for all resumes...

📊 ATS Score Statistics:
Mean Score: 62.7
Std Dev: 9.1
Min: 3.0
Max: 87.1

Pass Rate (score >= 60): 68.2%

Score Distribution:
count    2484.000000
mean       62.722658
std         9.134813
min         3.000000
25%        58.500000
50%        63.000000
75%        68.250000
max        87.083333
Name: ats_score, dtype: flo

In [5]:
# ============================================
# CLEANUP SCRIPT - Delete Generated Files
# Run this when you want to start fresh
# ============================================

import os

# List of files that get generated
generated_files = [
    'tfidf_vectorizer.pkl',
    'category_classifier.pkl',
    'skill_taxonomy.json',
    'ats_scorer.pkl',
    'ats_config.json',
    'job_embeddings.npy',
    'job_index.faiss',
    'job_metadata.pkl',
    'skill_job_mapping.json',
    'outcome_predictor.pkl',
    'outcome_config.json',
    'role_transition_graph.json',
    'transition_model.pkl',
    'salary_predictor.pkl'
]

print("🗑️ CLEANUP: Deleting generated files...")
print("-" * 40)

deleted = 0
not_found = 0

for filename in generated_files:
    if os.path.exists(filename):
        os.remove(filename)
        print(f"✅ Deleted: {filename}")
        deleted += 1
    else:
        print(f"⏭️ Not found (skip): {filename}")
        not_found += 1

print("-" * 40)
print(f"Summary: {deleted} deleted, {not_found} not found")
print("✅ Cleanup complete!")

🗑️ CLEANUP: Deleting generated files...
----------------------------------------
✅ Deleted: tfidf_vectorizer.pkl
✅ Deleted: category_classifier.pkl
✅ Deleted: skill_taxonomy.json
⏭️ Not found (skip): ats_scorer.pkl
⏭️ Not found (skip): ats_config.json
⏭️ Not found (skip): job_embeddings.npy
⏭️ Not found (skip): job_index.faiss
⏭️ Not found (skip): job_metadata.pkl
⏭️ Not found (skip): skill_job_mapping.json
⏭️ Not found (skip): outcome_predictor.pkl
⏭️ Not found (skip): outcome_config.json
⏭️ Not found (skip): role_transition_graph.json
⏭️ Not found (skip): transition_model.pkl
⏭️ Not found (skip): salary_predictor.pkl
----------------------------------------
Summary: 3 deleted, 11 not found
✅ Cleanup complete!
